# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 4 — Scholarly Chronology Acquisition (bug-fixed).**

Phase 3 established the corpus architecture and resolved the main Herrera edition question. Phase 4 begins the **primary composition-time backbone** with sources that provide defensible scholarly chronology. We still do **not** build semantic networks or choose temporal windows.


## What Phase 3 established

- `H.txt` is the **1582 H textual layer**: all 78 recovered H sonnets occur in `Herrera_Sonetos`.
- `P2.txt` is the **1619 posthumous textual layer**: `AN_SonetosP2` overlaps strongly with it, but is not identical.
- `AN` and `Herrera` must therefore be treated as edition/textual layers, not as independent authors.
- **1582 and 1619 are circulation/edition dates, not composition dates.**
- The primary historical axis remains composition/scholarly chronology; circulation is secondary robustness evidence.

The current goal is to attach poem-level chronology only where scholarly evidence supports it, with explicit provenance and uncertainty.


In [ ]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "navarro_tei": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "gongora_scholarly": (
        "https://github.com/gongoradigital/gongoraobra.git",
        "3beadeecc059a7cc48499dc2683bb378a2630978",
    ),
}
ROOT = Path("/content/gasr_phase4_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)], check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit], check=True)
    got = subprocess.check_output(["git","-C",str(dst),"rev-parse","HEAD"], text=True).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k,v in SOURCES.items()}
N = paths["navarro_tei"]
G = paths["gongora_scholarly"]
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

def local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def el_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def years_1580_1626(s):
    return sorted(set(
        int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)", str(s))
        if 1580 <= int(x) <= 1626
    ))

print("Pinned sources ready")
for k,v in SOURCES.items():
    print(f"  {k}: {v[1]}")
print("Python", sys.version.split()[0], "| pandas", pd.__version__)


## 01. Rebuild the Navarro poem backbone

Only poem identity, text and bibliographic provenance are rebuilt here. No TEI witness year is promoted to composition time.


In [ ]:
rows, parse_errors = [], []
xml_files = sorted(N.rglob("*.xml"))

for fp in xml_files:
    try:
        root = ET.parse(fp).getroot()
    except Exception as e:
        parse_errors.append((str(fp), repr(e)))
        continue

    lines = [el_text(x) for x in root.iter() if local(x.tag) == "l"]
    lines = [x for x in lines if x]
    if not lines:
        continue

    titles = [el_text(x) for x in root.iter() if local(x.tag) == "title" and el_text(x)]
    authors = [el_text(x) for x in root.iter() if local(x.tag) in {"author","name"} and el_text(x)]
    bibls = [el_text(x) for x in root.iter() if local(x.tag) in {"bibl","witness"} and el_text(x)]

    author_dir = fp.parent.name
    text = "\n".join(lines)
    rows.append({
        "n_id": f"{author_dir}::{fp.name}",
        "author_dir": author_dir,
        "author_tei": authors[0] if authors else "",
        "title": titles[0] if titles else "",
        "n_lines": len(lines),
        "text": text,
        "signature": norm(text),
        "source_bibl": " | ".join(bibls[:4]),
        "source_file": str(fp.relative_to(N)),
    })

n = pd.DataFrame(rows)
assert len(parse_errors) == 0, parse_errors[:5]

priority_A = {
    "GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa",
    "JuanDeArguijo","JuanDeJauregui","LuisCarrilloySotomayor","Cervantes",
    "Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"
}

print(f"Navarro poems: {len(n):,} | author folders: {n.author_dir.nunique():,}")
print("Priority-A poem counts:")
display(
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size()
    .sort_values(ascending=False)
    .rename("poems").reset_index()
)


## 02. Temporal master candidate

Primary chronology is composition/scholarly chronology. Publication, witness and edition dates remain separate evidence.

Confidence:
- **A**: explicit or strongly historically anchored scholarly year;
- **B**: defensible scholarly interval or high-quality text linkage to a dated scholarly corpus;
- **C**: broad/contested interval requiring sensitivity analysis;
- **unassigned**: no defensible composition-time evidence yet.


In [ ]:
temporal = n[["n_id","author_dir","title","source_file"]].copy()

for c in ["composition_min","composition_max","circulation_year"]:
    temporal[c] = pd.Series([pd.NA]*len(temporal), dtype="Int64")

temporal["temporal_confidence"] = "unassigned"
temporal["temporal_basis"] = ""
temporal["temporal_source"] = ""
temporal["chronology_status"] = "undated"

def assign_date_scalar(n_ids, lo, hi, confidence, basis, source):
    ids = set(n_ids)
    mask = temporal.n_id.isin(ids)
    assert mask.sum() == len(ids), (len(ids), int(mask.sum()))
    assert (temporal.loc[mask, "chronology_status"] == "undated").all(), "Attempted temporal overwrite."
    temporal.loc[mask, "composition_min"] = int(lo)
    temporal.loc[mask, "composition_max"] = int(hi)
    temporal.loc[mask, "temporal_confidence"] = confidence
    temporal.loc[mask, "temporal_basis"] = basis
    temporal.loc[mask, "temporal_source"] = source
    temporal.loc[mask, "chronology_status"] = "dated_current_sprint"

print("Temporal schema initialized:", len(temporal), "poems")


## 03. Góngora: scholarly chronological corpus

We use the pinned `gongoradigital/gongoraobra` TEI as an independent scholarly chronology. Years are extracted from the structural context of poem divisions. Only uniquely resolved years are eligible for the primary chronology, and only after poem-level text linkage to Navarro.


In [ ]:
gfile = G / "gongora_obra-poetica.xml"
assert gfile.exists(), gfile
groot = ET.parse(gfile).getroot()
parent = {child: par for par in groot.iter() for child in par}

poem_divs = []
for el in groot.iter():
    xid = el.attrib.get(XML_ID, "")
    if local(el.tag) == "div" and xid.lower().startswith("poem"):
        ls = [x for x in el.iter() if local(x.tag) == "l"]
        if ls:
            poem_divs.append(el)

def shallow_years(el):
    vals = list(el.attrib.values())
    if el.text:
        vals.append(el.text)
    for ch in list(el):
        if local(ch.tag) in {"head","date","label"}:
            vals.append(el_text(ch))
        if ch.tail:
            vals.append(ch.tail)
    ys = []
    for v in vals:
        ys += years_1580_1626(v)
    return ys

def ancestor_years(el, max_steps=6):
    ys, cur = [], el
    for _ in range(max_steps):
        ys += shallow_years(cur)
        cur = parent.get(cur)
        if cur is None:
            break
    return sorted(set(ys))

grows = []
for el in poem_divs:
    ls = [el_text(x) for x in el.iter() if local(x.tag) == "l"]
    ls = [x for x in ls if x]
    ys = ancestor_years(el)
    text = "\n".join(ls)
    grows.append({
        "g_id": el.attrib.get(XML_ID, ""),
        "g_n": el.attrib.get("n", ""),
        "n_lines": len(ls),
        "text": text,
        "signature": norm(text),
        "year_candidates": ";".join(map(str, ys)),
        "scholarly_year": ys[0] if len(ys) == 1 else pd.NA,
        "year_status": "unique" if len(ys) == 1 else ("ambiguous" if len(ys) > 1 else "missing"),
    })

g = pd.DataFrame(grows)
g["scholarly_year"] = pd.array(g["scholarly_year"], dtype="Int64")

print(f"Góngora XML poem divisions: {len(g):,}")
print("Line-count distribution (top):")
display(g.n_lines.value_counts().head(12).rename_axis("lines").reset_index(name="poems"))
print("Year extraction status:")
display(g.year_status.value_counts().rename_axis("status").reset_index(name="poems"))
print("Chronology range among uniquely resolved years:",
      g.scholarly_year.dropna().min(), "–", g.scholarly_year.dropna().max())
display(g[["g_id","g_n","n_lines","year_candidates","scholarly_year"]].head(20))


### 03.1 Link scholarly Góngora poems to Navarro sonnets

Exact normalized-text matches are accepted first. Remaining Navarro sonnets are compared only with 14-line scholarly poems. Fuzzy links at ≥0.98 are provisional and are rejected if they create a target collision.


In [ ]:
ng = n[n.author_dir.eq("Gongora")].copy()
g14 = g[g.n_lines.eq(14)].copy()

sig_to_gids = defaultdict(list)
for r in g14.itertuples(index=False):
    sig_to_gids[r.signature].append(r.g_id)

g_by_id = g14.set_index("g_id")
candidates = []

for r in ng.itertuples(index=False):
    exact_ids = sig_to_gids.get(r.signature, [])
    if len(exact_ids) == 1:
        gid = exact_ids[0]
        candidates.append({
            "n_id": r.n_id, "g_id": gid, "method": "exact",
            "score": 1.0, "preaccept": True
        })
        continue

    best_gid, best_score = None, -1.0
    for gr in g14.itertuples(index=False):
        sc = SequenceMatcher(None, r.signature, gr.signature).ratio()
        if sc > best_score:
            best_gid, best_score = gr.g_id, sc

    candidates.append({
        "n_id": r.n_id, "g_id": best_gid, "method": "fuzzy_provisional",
        "score": best_score, "preaccept": bool(best_score >= 0.98)
    })

glink = pd.DataFrame(candidates)
collision_ids = set(
    glink[glink.preaccept].groupby("g_id").size().loc[lambda s: s > 1].index
)
glink["accept"] = glink.preaccept & ~glink.g_id.isin(collision_ids)

glink = glink.merge(
    g14[["g_id","scholarly_year","year_status"]],
    on="g_id", how="left"
)

accepted = glink[glink.accept].copy()
accepted_dated = accepted[
    accepted.scholarly_year.notna() & accepted.year_status.eq("unique")
].copy()

for r in accepted_dated.itertuples(index=False):
    confidence = "A" if r.method == "exact" else "B"
    basis = (
        "scholarly_chronology_year_exact_text_link"
        if r.method == "exact"
        else "scholarly_chronology_year_high_similarity_link"
    )
    assign_date_scalar(
        [r.n_id], int(r.scholarly_year), int(r.scholarly_year),
        confidence, basis,
        "Cátedra Góngora / Carreira chronological corpus (gongoradigital/gongoraobra)"
    )

print("Navarro Góngora sonnets:", len(ng))
print("Exact links:", int((glink.method == "exact").sum()))
print("Fuzzy >= .98 candidates:", int(((glink.method == "fuzzy_provisional") & glink.preaccept).sum()))
print("Accepted one-to-one links:", int(glink.accept.sum()))
print("Accepted links with unique scholarly year:", len(accepted_dated))
print("Colliding scholarly targets excluded:", len(collision_ids))
display(glink.groupby("method").score.describe())
display(glink.sort_values("score").head(15)[["n_id","g_id","method","score","accept"]])


## 04. Garcilaso: conservative Lapesa/Rivers chronology seed

**Bug fix:** Navarro's Garcilaso titles are bare Roman numerals (`-I-`, `-II-`, …), not strings of the form `Soneto I`. We therefore parse the Roman title directly and independently recover the Arabic number from the filename. The two numbering systems must agree poem by poem before any chronology is assigned.


In [ ]:
roman_vals = {"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}

def roman_to_int(s):
    total, prev = 0, 0
    for ch in reversed(str(s).upper()):
        v = roman_vals.get(ch, 0)
        total += -v if v < prev else v
        prev = max(prev, v)
    return total

gar = n[n.author_dir.eq("GarcilasoDeLaVega")].copy()

# Navarro titles are formatted as -I-, -II-, ..., -XXXVIII-.
gar["sonnet_roman"] = gar.title.str.extract(
    r"^\s*-\s*([IVXLCDM]+)\s*-\s*$", expand=False
)
gar["title_no"] = gar.sonnet_roman.map(
    lambda x: roman_to_int(x) if pd.notna(x) else pd.NA
).astype("Int64")

# Independent cross-check from source filename ..._01.xml, ..._38.xml.
gar["file_no"] = pd.to_numeric(
    gar.n_id.str.extract(r"_(\d+)\.xml$", expand=False),
    errors="coerce"
).astype("Int64")

print("Garcilaso Navarro records:", len(gar))
print("Roman-title numbers parsed:", int(gar.title_no.notna().sum()))
print("Filename numbers parsed:", int(gar.file_no.notna().sum()))
print("Title/filename agreements:", int((gar.title_no == gar.file_no).fillna(False).sum()))

assert len(gar) == 38, f"Expected 38 Garcilaso sonnets, found {len(gar)}."
assert gar.title_no.notna().all(), "Some Garcilaso Roman titles were not parsed."
assert gar.file_no.notna().all(), "Some Garcilaso filenames were not parsed."
assert (gar.title_no == gar.file_no).all(), "Garcilaso title and filename numbering disagree."
assert gar.title_no.duplicated().sum() == 0, "Duplicate canonical sonnet numbers require inspection."

gar["sonnet_no"] = gar["title_no"]
display(gar[["n_id","title","sonnet_roman","sonnet_no"]].sort_values("sonnet_no"))
print("Garcilaso numbering integrity: PASSED (38/38).")


In [ ]:
GAR_CHRONOLOGY = {
    **{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
    25:(1534,1535,"B","scholarly_interval"),
    33:(1535,1535,"A","historically_anchored_scholarly_year"),
    35:(1535,1535,"A","historically_anchored_scholarly_year"),
    **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]},
}

gassign = []
for no,(lo,hi,conf,basis) in GAR_CHRONOLOGY.items():
    z = gar[gar.sonnet_no.eq(no)]
    if len(z) != 1:
        print("WARNING: canonical number not uniquely found:", no, len(z))
        continue
    nid = z.iloc[0].n_id
    assign_date_scalar(
        [nid], lo, hi, conf, basis,
        "Rafael Lapesa chronology as summarized/discussed by CVC (E. L. Rivers) and AISO scholarship"
    )
    gassign.append({
        "sonnet_no": no, "n_id": nid,
        "composition_min": lo, "composition_max": hi,
        "temporal_confidence": conf, "temporal_basis": basis
    })

gar_chron = pd.DataFrame(gassign).sort_values("sonnet_no")
print("Garcilaso conservatively dated this sprint:", len(gar_chron), "/", len(gar))
print("Garcilaso still unassigned:", len(gar) - len(gar_chron))
display(gar_chron)


## 05. Current Priority-A temporal coverage

This is **not yet the final historical corpus**. It tells us how much primary-axis chronology is defensible after the first two scholarly acquisitions and which authors must be researched next.


In [ ]:
coverage = (
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size().rename("total_poems").to_frame()
    .join(
        temporal[temporal.chronology_status.ne("undated")]
        .groupby("author_dir").size().rename("dated_current")
    )
    .fillna(0)
)
coverage["dated_current"] = coverage.dated_current.astype(int)
coverage["coverage_pct"] = (100 * coverage.dated_current / coverage.total_poems).round(1)
coverage = coverage.sort_values(
    ["coverage_pct","total_poems"], ascending=[False,False]
).reset_index()
display(coverage)

print("Confidence distribution among currently dated poems:")
display(
    temporal[temporal.chronology_status.ne("undated")]
    .temporal_confidence.value_counts()
    .rename_axis("confidence").reset_index(name="poems")
)

dated = temporal[temporal.chronology_status.ne("undated")].copy()
assert dated.composition_min.notna().all() and dated.composition_max.notna().all()
assert (dated.composition_min.astype(int) <= dated.composition_max.astype(int)).all()
assert not dated.temporal_basis.str.contains(
    "publication|witness|edition", case=False, regex=True
).any(), "Publication/witness/edition evidence leaked into the composition-time axis."
print("Temporal integrity checks: PASSED")


## 06. Runtime exports

The CSVs below are derived diagnostics. The notebook plus pinned source commits remain the provenance record. After review, the stable temporal table can later move to `data/derived/` or `results/tables/`.


In [ ]:
OUT = Path("/content/gasr_phase4_outputs")
OUT.mkdir(exist_ok=True)

temporal.to_csv(OUT/"temporal_master_candidate.csv", index=False)
glink.to_csv(OUT/"gongora_match_diagnostics.csv", index=False)
gar_chron.to_csv(OUT/"garcilaso_chronology_seed.csv", index=False)
coverage.to_csv(OUT/"priority_A_temporal_coverage.csv", index=False)

print("Runtime outputs:")
for p in sorted(OUT.glob("*.csv")):
    print(" ", p)
print()
print("PHASE 4 CHECKPOINT")
print("------------------")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: inspect Góngora scholarly-year linkage and Garcilaso coverage;")
print("then extend defensible dating to the remaining Priority-A authors.")
